# Round-1 Annotation Pipeline — Kaggle Runner

**Model**: `Qwen/Qwen3.5-2B` (natively multimodal, 2B params, ~2 GB VRAM in 4-bit)  
**GPU**: T4 — 16 GB VRAM (Kaggle free tier)

## Before running — checklist

| # | Step | Where |
|---|------|-------|
| 1 | Enable **GPU T4** | Session options → Accelerator → GPU T4 |
| 2 | Enable **Internet** | Session options → Internet → On |
| 3 | Add `HF_TOKEN` secret | Add-ons → Secrets → New secret |
| 4 | Attach your dataset | Add data → Your datasets |
| 5 | Set `REPO_URL` and `DATASET_SLUG` in **Cell 4** | — |

> **Internet must be ON** — the notebook clones the repo from GitHub and downloads  
> the model weights (~4 GB) from HuggingFace on first run.

The pipeline saves a checkpoint after every batch — re-running the notebook resumes from where it stopped.

## 1. Install dependencies

`Qwen3.5-2B` requires `transformers >= 4.51.0`. We install from the GitHub source to get the latest stable version.

In [1]:
import subprocess, sys

def _pip(*args):
    """Run pip and print output; raise on failure."""
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:] if r.stdout else "")
        raise RuntimeError(f"pip install failed:\n{r.stderr[-2000:]}")

_pip("git+https://github.com/huggingface/transformers")  # >= 4.51.0 for Qwen3.5-2B
_pip("accelerate>=0.30.0", "bitsandbytes>=0.43.0",
     "pydantic>=2.0", "pyyaml", "tqdm", "Pillow")
print("All dependencies installed.")

All dependencies installed.


## 2. Verify GPU

In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable GPU in Session options → Accelerator → GPU T4.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

import shutil
free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"Disk free (/kaggle/working) : {free_gb:.1f} GB")
if free_gb < 6:
    print("WARNING: Less than 6 GB free — model download (~4.2 GB) may fail.")

GPU  : Tesla T4
VRAM : 14.6 GB
Disk free (/kaggle/working) : 19.5 GB


## 3. Verify Qwen3.5-2B transformers support

In [3]:
import transformers
print(f"Transformers version : {transformers.__version__}")

try:
    from transformers import AutoModelForImageTextToText
    print("AutoModelForImageTextToText : OK")
except ImportError as e:
    raise RuntimeError(
        f"AutoModelForImageTextToText not found: {e}\n"
        "Re-run Cell 1 to reinstall transformers from source."
    )

Transformers version : 5.6.0.dev0
AutoModelForImageTextToText : OK


## 4. Environment configuration

**Edit these two variables before running:**

In [4]:
import os

print("Đang tìm file json...")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == "final_50_samples.json":
            print(f"👉 TÌM THẤY RỒI! Đường dẫn chuẩn là: {os.path.join(dirname, filename)}")

Đang tìm file json...
👉 TÌM THẤY RỒI! Đường dẫn chuẩn là: /kaggle/input/datasets/kghangco/sarcasm-detection-50samples/final_50_samples.json


In [5]:
import os
from pathlib import Path

REPO_URL     = "https://github.com/nhatvu205/vi-multimodal-sacarsm-detection-on-social-media.git"

KAGGLE_WORKING = Path("/kaggle/working")
REPO_DIR       = KAGGLE_WORKING / "vi-multimodal-sacarsm-detection-on-social-media"
ROUND1_DIR     = REPO_DIR / "round-1-annotation"
OUTPUT_DIR     = KAGGLE_WORKING / "outputs"

PKG_DIR = Path("/kaggle/input/datasets/kghangco/sarcasm-detection-50samples/")
INPUT_JSON = PKG_DIR / "final_50_samples.json"
IMAGES_DIR = PKG_DIR / "images"   

HF_CACHE_DIR = KAGGLE_WORKING / "hf_cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)

print(f"Input JSON   : {INPUT_JSON}")
print(f"Images dir   : {IMAGES_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")

# Sanity checks
if not INPUT_JSON.exists():
    raise FileNotFoundError(f"Input JSON not found: {INPUT_JSON}")
if not IMAGES_DIR.exists():
    raise FileNotFoundError(f"Images folder not found: {IMAGES_DIR}")
print(f"Images available: {len(list(IMAGES_DIR.iterdir()))}")

# Load JSON
import json
raw = INPUT_JSON.read_text(encoding="utf-8").strip()
if raw.startswith("["):
    all_records = json.loads(raw)
else:
    all_records = [json.loads(l) for l in raw.splitlines() if l.strip()]

print(f"Records loaded: {len(all_records)}")

Input JSON   : /kaggle/input/datasets/kghangco/sarcasm-detection-50samples/final_50_samples.json
Images dir   : /kaggle/input/datasets/kghangco/sarcasm-detection-50samples/images
Output dir   : /kaggle/working/outputs
Images available: 50
Records loaded: 50


## 5. Clone repo and set Python path

In [6]:
import subprocess, sys

if REPO_URL and not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} ...")
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{r.stderr}")
    print("Clone complete.")
elif REPO_DIR.exists():
    print(f"Repo present at {REPO_DIR} — pulling latest ...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    print("REPO_URL is None — using code already in REPO_DIR.")

if str(ROUND1_DIR) not in sys.path:
    sys.path.insert(0, str(ROUND1_DIR))
print(f"sys.path includes: {ROUND1_DIR}")

Cloning https://github.com/nhatvu205/vi-multimodal-sacarsm-detection-on-social-media.git ...
Clone complete.
sys.path includes: /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/round-1-annotation


## 6. Load HuggingFace token from Kaggle Secrets

In [7]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if HF_TOKEN:
        print("HF_TOKEN loaded from environment variable.")
    else:
        print("HF_TOKEN not set — will proceed without it (public models only).")

HF_TOKEN not set — will proceed without it (public models only).


## 7. Locate input data

In [8]:
import json
import random
from pathlib import Path

if not INPUT_JSON.exists():
    for candidate in ["data.jsonl", "data-sample.json", "annotations.json"]:
        p = DATASET_DIR / candidate
        if p.exists():
            INPUT_JSON = p
            break

if not INPUT_JSON.exists():
    raise FileNotFoundError(
        f"No input JSON found in {DATASET_DIR}.\n"
        "Set INPUT_JSON manually to the correct path."
    )

raw = INPUT_JSON.read_text(encoding="utf-8").strip()

if raw.startswith("["):
    all_records = json.loads(raw)
    is_jsonl = False
else:
    all_records = [json.loads(l) for l in raw.splitlines() if l.strip()]
    is_jsonl = True

n_records = len(all_records)

print(f"Original file : {INPUT_JSON}")
print(f"Records       : {n_records}")

Original file : /kaggle/input/datasets/kghangco/sarcasm-detection-50samples/final_50_samples.json
Records       : 50


In [9]:
import json
from pathlib import Path

# Patch all_records
CORRECT_BASE_PATH = "/kaggle/input/datasets/kghangco/sarcasm-detection-50samples/images/"
for record in all_records:
    if "image_path" in record:
        record["image_path"] = CORRECT_BASE_PATH + Path(record["image_path"]).name

In [10]:
import src.llm_judge as lj
from PIL import Image
from typing import Optional

def _open_image_patched(image_path: str, max_pixels: int = 500_000) -> Optional[Image.Image]:
    filename = Path(image_path).name
    candidate = IMAGES_DIR / filename
    if candidate.exists():
        img = Image.open(candidate).convert("RGB")
        return lj._resize_image(img, max_pixels)
    lj.logger.warning("Image not found: %s", candidate)
    return None

lj._open_image = _open_image_patched
print("Patched _open_image ✓")

Patched _open_image ✓


In [11]:
test_path = all_records[0]["image_path"]
img = lj._open_image(test_path)
print("Image loaded:", img.size if img else "FAILED")

Image loaded: (816, 612)


In [12]:
# Ghi ra file mới
patched_json = Path("/kaggle/working/patched_input.json")
patched_json.write_text(json.dumps(all_records, ensure_ascii=False), encoding="utf-8")

# Verify
print(all_records[0]["image_path"])
print(Path(all_records[0]["image_path"]).exists())

/kaggle/input/datasets/kghangco/sarcasm-detection-50samples/images/post0092_img03.jpg
True


## 9. Verify prompt files

In [13]:
for fname in ["prompt.txt", "few-short-examples.txt"]:
    p = ROUND1_DIR / "prompts" / fname
    status = "OK" if p.exists() else "MISSING"
    print(f"{fname:30s} : {status}")
    if not p.exists():
        raise FileNotFoundError(f"Prompt file missing: {p}")

prompt.txt                     : OK
few-short-examples.txt         : OK


In [14]:
!cat /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/round-1-annotation/prompts/prompt.txt

You are an annotator for multimodal sarcasm detection in Vietnamese social media posts.

Your task is to determine whether a given post is non-sarcastic (label 0) or sarcastic (label 1), 
by analyzing three modalities: text, images, and emoji. Not every input contains images and/or emoji.

Reason through all checks in order; do not skip any, even if the answer seems obvious.
Return exactly one valid JSON object and nothing else — no explanation, no markdown, no text outside the JSON.
All reasoning fields must be written in Vietnamese.
All JSON keys and enum values must be in English exactly as shown in the output schema.

Input:
[TEXT]
{text}

[IMAGES]
{images}

---

=== ĐỊNH NGHĨA NHÃN ===

Nhãn 0 = Non-sarcastic: Text, ảnh, emoji nhất quán; không có conflict ẩn; tất cả phương thức cùng chiều; nghĩa literal đúng với thực tế.
Nhãn 1 = Sarcastic: Nghĩa bề mặt KHÁC ý thật; có ít nhất một conflict rõ ràng. Nếu có mỉa mai, nó thường rơi vào 1 trong 7 nhóm sau (sẽ dùng để phân tích trong ph

## 10. Quick model smoke-test (optional)

Load `Qwen3.5-2B`, run one text inference, then free GPU memory.  
**Skip this cell** if you want to go straight to the pipeline and save session time.

In [15]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
print(f"Loading {MODEL_ID} in 4-bit (this downloads ~2 GB on first run) ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
smoke_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN or None,
)
smoke_proc = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN or None)

test_msgs = [{"role": "user", "content": [{"type": "text", "text": "Reply with the single word: OK"}]}]
inputs = smoke_proc.apply_chat_template(
    test_msgs, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt"
)
# Move all tensor values to the model's device
device = next(smoke_model.parameters()).device
inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

with torch.no_grad():
    out_ids = smoke_model.generate(**inputs, max_new_tokens=10, do_sample=False)

trimmed = out_ids[0][inputs["input_ids"].shape[-1]:]
reply = smoke_proc.decode(trimmed, skip_special_tokens=True)
print(f"Smoke-test reply: '{reply}'")

del smoke_model, smoke_proc, inputs, out_ids
torch.cuda.empty_cache()
print("Model unloaded — GPU memory released for pipeline.")

Loading Qwen/Qwen3-VL-4B-Instruct in 4-bit (this downloads ~2 GB on first run) ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Smoke-test reply: 'OK'
Model unloaded — GPU memory released for pipeline.


## 11. Run the annotation pipeline

In [16]:
from src.pipeline_round1 import run_pipeline

CONFIG_PATH = str(ROUND1_DIR / "configs" / "round1.yaml")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Round-1 annotation pipeline (SAMPLED MODE)")
print(f"  Model  : Qwen/Qwen3.5-2B (4-bit)")
print(f"  Input  : {INPUT_JSON} ({n_records} records)")
print(f"  Config : {CONFIG_PATH}")
print(f"  Output : {OUTPUT_DIR}")
print("=" * 60)

run_pipeline(
    input_data=str(patched_json),
    config_path=CONFIG_PATH,
    output_dir=str(OUTPUT_DIR),
    hf_token=HF_TOKEN or None,
)

print("Pipeline finished.")

Round-1 annotation pipeline (SAMPLED MODE)
  Model  : Qwen/Qwen3.5-2B (4-bit)
  Input  : /kaggle/input/datasets/kghangco/sarcasm-detection-50samples/final_50_samples.json (50 records)
  Config : /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/round-1-annotation/configs/round1.yaml
  Output : /kaggle/working/outputs
2026-04-15T13:44:50Z | INFO     | src.pipeline_round1 | === Round-1 Pipeline Start ===
2026-04-15T13:44:50Z | INFO     | src.pipeline_round1 | Model: Qwen/Qwen3.5-2B | device: cuda | 4bit: True | max_img_px: 500000
2026-04-15T13:44:50Z | INFO     | src.pipeline_round1 | RouterConfig: RouterConfig(random_audit_rate=0.1, seed=42)


2026-04-15T13:44:50Z | INFO     | src.loaders | Loaded 50 input records from /kaggle/working/patched_input.json
2026-04-15T13:44:50Z | INFO     | src.pipeline_round1 | Running LLM on 50 records (0 already cached)


Overall LLM progress:   0%|          | 0/50 [00:00<?, ?rec/s]

2026-04-15T13:44:50Z | INFO     | src.pipeline_round1 | Batch 1/7 (8 records)
2026-04-15T13:44:50Z | INFO     | src.llm_judge | Loading model: Qwen/Qwen3.5-2B  (device=cuda, 4bit=True)


config.json: 0.00B [00:00, ?B/s]

...


LLM judging:   0%|          | 0/2 [00:00<?, ?rec/s][transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


2026-04-15T14:48:31Z | WARNING  | src.llm_judge | Bad JSON for id=48, retrying repair. Raw: {
  "reasoning": "First check: Valid, không có thông tin PII. Second check: Text 'T đi Huế xong t thấy cái j cũng rẻ, hay t dọn ra đây sống 1 tgian nhé' là lời khen ngợi, nhưng có sự mâu thuẫn với hìn
2026-04-15T14:48:31Z | ERROR    | src.llm_judge | Repair failed for id=48: string indices must be integers, not 'str'



LLM judging:  50%|█████     | 1/2 [01:50<01:50, 110.34s/rec][transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.

Routing records: 100%|██████████| 50/50 [00:00<00:00, 50207.13rec/s]

2026-04-15T14:49:24Z | INFO     | src.fusion_router | Audit sampling: 6 auto-accepted candidates, 1 sampled (rate=0.10)


2026-04-15T14:49:24Z | INFO     | src.pipeline_round1 | Outputs written to /kaggle/working/outputs | all=50 | auto=5 | human=45 | bad=0
2026-04-15T14:49:24Z | INFO     | src.pipeline_round1 | === Round-1 Pipeline Complete ===
2026-04-15T14:49:24Z | INFO     | src.pipeline_round1 | auto_accept_rate=0.10 | human_queue_rate=0.90 | invalid=44
Pipeline finished.


## 12. Inspect results

In [17]:
stats_file = OUTPUT_DIR / "round1_stats.json"
if stats_file.exists():
    print(json.dumps(json.loads(stats_file.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))
else:
    print(f"Stats file not found: {stats_file}")

{
  "total_samples": 50,
  "processed_samples": 50,
  "bad_records": 0,
  "auto_accepted_count": 5,
  "human_queue_count": 45,
  "auto_accept_rate": 0.1,
  "human_queue_rate": 0.9,
  "invalid_count": 44,
  "audit_sampled_count": 1,
  "class_distribution_auto_accepted": {
    "sarcastic": 4,
    "non_sarcastic": 1
  },
  "route_reason_distribution": {
    "high_conf": 5,
    "uncertain": 23,
    "invalid_json": 21,
    "audit_sampled": 1
  },
  "difficulty_distribution": {
    "Easy": 18,
    "Hard": 0,
    "null": 32
  },
  "text_only_distribution": {
    "0": 3,
    "1": 5,
    "null": 42
  },
  "imageset_only_distribution": {
    "0": 4,
    "1": 0,
    "null": 46
  }
}


In [18]:
for label, fname in [("Auto-accepted", "round1_auto_accepted.jsonl"),
                     ("Human-queue",   "round1_human_queue.jsonl")]:
    f = OUTPUT_DIR / fname
    if not f.exists():
        print(f"{label}: file not found")
        continue
    lines = f.read_text(encoding="utf-8").splitlines()
    print(f"\n{label}: {len(lines)} records")
    for line in lines[:3]:
        rec = json.loads(line)
        print(f"  id={rec['id']} | label={rec['label_llm1']} | difficulty={rec['difficulty']} | route={rec['route_reason']}")


Auto-accepted: 5 records
  id=0 | label=1 | difficulty=Easy | route=high_conf
  id=11 | label=1 | difficulty=Easy | route=high_conf
  id=17 | label=0 | difficulty=Easy | route=high_conf

Human-queue: 45 records
  id=1 | label=INVALID | difficulty=Easy | route=uncertain
  id=2 | label=INVALID | difficulty=None | route=uncertain
  id=3 | label=INVALID | difficulty=None | route=invalid_json


## 13. Output file summary

Files are in `/kaggle/working/outputs/` and are committed automatically when you save the notebook run.

In [19]:
print("Output files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.name.startswith("."):
        continue
    print(f"  {f.name:45s}  {f.stat().st_size / 1024:8.1f} KB")

Output files:
  bad_records.jsonl                                   0.0 KB
  round1_all.jsonl                                   21.6 KB
  round1_auto_accepted.jsonl                          2.1 KB
  round1_human_queue.jsonl                           19.5 KB
  round1_stats.json                                   0.7 KB
